In [70]:
# # Initialize Otter
# import otter
# grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 
**Due date: See the [Calendar](https://htmlpreview.github.io/?https://github.com/UBC-CS/cpsc330/blob/master/docs/calendar.html).**

## Imports

In [71]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import classification_report

<br><br>

## Submission instructions
<hr>
rubric={points:2}

You will receive marks for correctly submitting this assignment. To submit this assignment, follow the instructions below:

- **You may work on this assignment in a group (group size <= 4) and submit your assignment as a group.** 
- Below are some instructions on working as a group.  
    - The maximum group size is 4. 
    - You can choose your own group members. 
    - Use group work as an opportunity to collaborate and learn new things from each other. 
    - Be respectful to each other and make sure you understand all the concepts in the assignment well. 
    - It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. [Here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members) are some instructions on adding group members in Gradescope.  
- Upload the .ipynb file to Gradescope.
- **If the .ipynb file is too big or doesn't render on Gradescope for some reason, also upload a pdf or html in addition to the .ipynb.** 
- Make sure that your plots/output are rendered properly in Gradescope.

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [72]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [73]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [74]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points:2}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words in `glove_wiki_vectors` embeddings. 

<div class="alert alert-warning">

Solution_1.1
    
</div>

In [75]:
glove_wiki_vectors.most_similar("classy")

[('stylish', 0.786424458026886),
 ('gutsy', 0.7500391006469727),
 ('tasteful', 0.7096198201179504),
 ('delightful', 0.6781403422355652),
 ('charming', 0.6778756976127625),
 ('unpretentious', 0.6721422672271729),
 ('easygoing', 0.6648709177970886),
 ('flashy', 0.6643825769424438),
 ('glamorous', 0.664241373538971),
 ('gorgeous', 0.6601454019546509)]

In [76]:
glove_wiki_vectors.most_similar("pomegranate")

[('apricot', 0.7174665331840515),
 ('mango', 0.7000153660774231),
 ('tamarind', 0.6881566047668457),
 ('pumpkin', 0.6772226691246033),
 ('pear', 0.6767832040786743),
 ('pineapple', 0.6580570340156555),
 ('papaya', 0.6526557803153992),
 ('cranberry', 0.6495745182037354),
 ('peach', 0.6447383165359497),
 ('watermelon', 0.6376970410346985)]

In [77]:
glove_wiki_vectors.most_similar("vancouver")

[('calgary', 0.8236873149871826),
 ('toronto', 0.8086004853248596),
 ('edmonton', 0.7792403697967529),
 ('ottawa', 0.7779433727264404),
 ('winnipeg', 0.7616495490074158),
 ('montreal', 0.7592548727989197),
 ('portland', 0.6879023313522339),
 ('phoenix', 0.6766419410705566),
 ('seattle', 0.6594581604003906),
 ('philadelphia', 0.6500991582870483)]

In [78]:
glove_wiki_vectors.most_similar("nostalgia")

[('nostalgic', 0.7496767640113831),
 ('angst', 0.6393096446990967),
 ('longing', 0.631801187992096),
 ('fascination', 0.6292052865028381),
 ('obsession', 0.6189970374107361),
 ('euphoria', 0.6159963011741638),
 ('sentimentality', 0.6133909821510315),
 ("'70s", 0.6125956177711487),
 ('bitterness', 0.6089754104614258),
 ('irony', 0.6079596877098083)]

In [79]:
glove_wiki_vectors.most_similar("sleep")

[('sleeping', 0.7516134977340698),
 ('breathing', 0.6675446629524231),
 ('asleep', 0.6665509343147278),
 ('awake', 0.6566905379295349),
 ('bed', 0.6524335741996765),
 ('breath', 0.6323226094245911),
 ('slept', 0.6264017820358276),
 ('waking', 0.6135101318359375),
 ('sick', 0.6024948358535767),
 ('eat', 0.5992845892906189)]

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points:2}

**Your tasks:**
1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [80]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

In [81]:
pair_similarity = [glove_wiki_vectors.similarity(pair[0], pair[1]) .item() for pair in word_pairs]
pair_similarity
    

[0.7000271677970886,
 0.5462760329246521,
 0.6432487964630127,
 0.7552732825279236,
 0.8798074722290039,
 0.07671945542097092]

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Stereotypes and biases in embeddings
rubric={points:6}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings or not by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore what kind of stereotypes and biases are encoded in these embeddings. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)   
2. Discuss your observations.

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

An example of using analogy to explore biases and stereotypes.  

In [82]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

In [83]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


An example of using similarity between words to explore biases and stereotypes.  

In [84]:
glove_wiki_vectors.similarity("white", "rich")

np.float32(0.44723597)

In [85]:
glove_wiki_vectors.similarity("black", "rich")

np.float32(0.51745194)

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Type your answer here, replacing this text._

In [86]:
analogy("white", "generous", "jewish")

white : generous :: jewish : ?


,Analogy word,Score
0,restitution,0.513825
1,needy,0.499154
2,charitable,0.498195
3,jews,0.494183
4,cherished,0.490759
5,non-jewish,0.473205
6,holocaust,0.462333
7,kibbutz,0.460420
8,haredi,0.459455
9,overdue,0.459035


In [87]:
analogy("asian", "math", "black")

asian : math :: black : ?


,Analogy word,Score
0,teacher,0.522524
1,classroom,0.519433
2,school,0.516746
3,graders,0.498161
4,reading,0.489991
5,taught,0.489203
6,mathematics,0.479179
7,teachers,0.478444
8,grade,0.476839
9,pencil,0.468640


In [88]:
glove_wiki_vectors.similarity("black", "crime")

np.float32(0.43576175)

In [89]:
glove_wiki_vectors.similarity("asian", "crime")

np.float32(0.28566787)

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Discussion
rubric={points:4}

**Your tasks:**
1. Based on your exploration above, comment on the overall quality of these pre-trained embeddings. 
2. In the lecture, we saw that our pre-trained word embedding model output an analogy that reinforced a gender stereotype. Give an example of how using such a model could cause harm in the real world.

<div class="alert alert-warning">

Solution_1.4
    
</div>

if this word vectorizer was used in, say an education app's chatbot, children could learn these harmful stereotypes

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Classification with pre-trained embeddings 
rubric={points:8}

In lecture 18, we saw that you can conveniently get word vectors with `spaCy` with `en_core_web_md` model. In this exercise, you'll use word embeddings in multi-class text classification task. We will use [HappyDB](https://www.kaggle.com/ritresearch/happydb) corpus which contains about 100,000 happy moments classified into 7 categories: *affection, exercise, bonding, nature, leisure, achievement, enjoy_the_moment*. The data was crowd-sourced via [Amazon Mechanical Turk](https://www.mturk.com/). The ground truth label is not available for all examples, and in this lab, we'll only use the examples where ground truth is available (~15,000 examples). 

- Download the data from [here](https://www.kaggle.com/ritresearch/happydb).
- Unzip the file and copy it under data/ directory in this homework directory.

The code below reads the data CSV (assuming that it's present in the current directory as *cleaned_hm.csv*),  cleans it up a bit, and splits it into train and test splits. 

**Your tasks:**

1. Train logistic regression with bag-of-words features (`CountVectorizer`) and show [classification report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) on the test set. 
2. Train logistic regression with average embedding representation extracted using spaCy and show [classification report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) on the test set. (You can refer to lecture 18 notes for this.)  

In [181]:
df = pd.read_csv("data/cleaned_hm.csv", index_col=0)
sample_df = df.dropna()
sample_df.head()

,wid,reflection_period,original_hm,cleaned_hm,modified,num_sentence,ground_truth_category,predicted_category
hmid,,,,,,,,
27676,206,24h,We had a serious talk with some friends of our...,We had a serious talk with some friends of our...,True,2,bonding,bonding
27678,45,24h,I meditated last night.,I meditated last night.,True,1,leisure,leisure
27697,498,24h,My grandmother start to walk from the bed afte...,My grandmother start to walk from the bed afte...,True,1,affection,affection
27705,5732,24h,I picked my daughter up from the airport and w...,I picked my daughter up from the airport and w...,True,1,bonding,affection
27715,2272,24h,when i received flowers from my best friend,when i received flowers from my best friend,True,1,bonding,bonding


In [182]:
sample_df = sample_df.rename(
    columns={"cleaned_hm": "moment", "ground_truth_category": "target"}
)

In [183]:
train_df, test_df = train_test_split(sample_df, test_size=0.3, random_state=123)
X_train, y_train = train_df["moment"], train_df["target"]
X_test, y_test = test_df["moment"], test_df["target"]

In [184]:
# X_train = pd.DataFrame(X_train)
# X_test = pd.DataFrame(X_test)

You need `spacy` to run the code below. If it's not in your course conda environment, you need to install it. 

> conda install -c conda-forge spacy

You also need to download the following language model. 

> python -m spacy download en_core_web_md

<div class="alert alert-warning">

Solution_1.5
    
</div>

_Type your answer here, replacing this text._

In [185]:
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(min_df=5, max_df=100, stop_words="english")

In [186]:
X_train_t = vec.fit_transform(X_train)
X_test_t = vec.transform(X_test)

In [187]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier()
dummy.fit(X_train_t, y_train)
dummy.score(X_test_t, y_test)

0.33577159037281734

In [188]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_t, y_train)
lr.score(X_test_t, y_test)

0.5910806984426616

In [189]:
print(classification_report(lr.predict(X_test_t), y_test))

                  precision    recall  f1-score   support

     achievement       0.71      0.67      0.69      1368
       affection       0.75      0.56      0.64      1899
         bonding       0.35      0.54      0.42       313
enjoy_the_moment       0.29      0.42      0.34       322
        exercise       0.53      0.85      0.65        46
         leisure       0.35      0.58      0.44       245
          nature       0.41      0.64      0.50        45

        accuracy                           0.59      4238
       macro avg       0.48      0.61      0.53      4238
    weighted avg       0.64      0.59      0.61      4238



<!-- END QUESTION -->

<br><br>

In [190]:
import spacy
nlp = spacy.load("en_core_web_md")

In [191]:
def preprocess(doc, min_token_len=2, irrelevant_pos=["ADV", "PRON", "CCONJ", "PUNCT", "PART", "DET", "ADP", "SPACE"]):
    clean_text = []
    
    for token in doc:
        if (token.is_stop == False and len(token) > min_token_len and token.pos_ not in irrelevant_pos):
            lemma = token.lemma_
            clean_text.append(lemma.lower())
    return " ".join(clean_text)

In [192]:
X_train = X_train.to_frame(name="text")
X_test = X_test.to_frame(name="text")

In [193]:
X_train.shape

(9887, 1)

In [194]:
X_train["text_pp"] = [preprocess(text) for text in nlp.pipe(X_train["text"])]
X_test["text_pp"] = [preprocess(text) for text in nlp.pipe(X_test["text"])]

In [198]:
X_train.head()

,text,text_pp
hmid,,
50152,I sorted and organized all my tax documents in...,sort organize tax document preparation file taxis
97451,"Last week my dog had 8 puppies, adorable !",week dog puppy adorable
77308,I received my computer certification.,receive computer certification
41407,An event that made me happy was seeing my girl...,event happy see girlfriend yesterday go hike g...
74665,My daughter had her 6 year old birthday.,daughter year old birthday


In [199]:
X_train = X_train.drop(columns=["text"])
X_test = X_test.drop(columns=["text"])

In [200]:
X_train.head()

,text_pp
hmid,
50152,sort organize tax document preparation file taxis
97451,week dog puppy adorable
77308,receive computer certification
41407,event happy see girlfriend yesterday go hike g...
74665,daughter year old birthday


In [201]:
vec2 = CountVectorizer(stop_words="english")
X_train_t2 = vec2.fit_transform(X_train["text_pp"]) 
X_test_t2 = vec2.transform(X_test["text_pp"])

In [203]:
dummy2 = DummyClassifier()
dummy2.fit(X_train_t2, y_train)
dummy2.score(X_test_t2, y_test)

0.33577159037281734

In [204]:
lr2 = LogisticRegression(max_iter=1000)
lr2.fit(X_train_t2, y_train)
lr2.score(X_test_t2, y_test)

0.8204341670599339

<!-- BEGIN QUESTION -->

### 1.6 Discussion
rubric={points:6}

**Your tasks:**
1. Briefly explain the difference between using `CountVectorizer` vs. average-embedding approach for text classification.  
2. Which representation among these two would be more interpretable? Why?   
3. Are we using any transfer learning here? If yes, are you observing any benefits of transfer learning? Briefly discuss. 

<div class="alert alert-warning">

Solution_1.6
    
</div>

1. count vectorizer takes word counts from the raw text. average embedding strips raw text to only the most telling words (which we tried to do with the hyperparamters with the first approach), then applied count vectorizer to this new stripped raw text column
2. count vectorizer would make more sense because it is simply turning words into counts. average-embedding requires one extra step that eliminates some clarity by removing a lot of regular english words, making it less interpretable
3. yes, we are transferring the learning from the spacy library to our current dataset. accuracy improved by over twenty percent! 

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The overarching goal of topic modeling is understanding high-level themes in a large collection of texts in an unsupervised way. 

In this exercise you will explore topics in a subset of `scikit-learn`'s [20 newsgroups text dataset](https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [99]:
from sklearn.datasets import fetch_20newsgroups

In [100]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

KeyboardInterrupt: 

In [ ]:
newsgroups_train.target_names

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points:8}

In this exercise you'll prepare the data for topic modeling using [spaCy](https://spacy.io/). Preprocessing is a crucial step before training an LDA model and it markedly affects topic modeling results. So let's carry out preprocessing. 

**Your tasks:** 

Write code to carry out preprocessing of the "text" column in the dataframe above and store the preprocessed text in a new column called "text_pp" in the dataframe. 


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are more informative and which ones are less informative for the given task. Some common text preprocessing steps for topic modeling are: 
- getting rid of slashes or other weird characters
- sentence segmentation and tokenization      
- getting rid of urls and email addresses
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization        

You might have to go back and forth between the preprocessing and topic modeling and interpretation steps in the next exercises. 

> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://spacy.io/api/annotation/#pos-en) is the list of part-of-speech tags used in spaCy. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you're done with the preprocessing part, you might want to save the preprocessed data so that you don't run the preprocessing part every time you run the notebook. 


<div class="alert alert-warning">

Solution_2_1
    
</div>

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Justification
rubric={points:2}

**Your tasks:**

Outline the preprocessing steps you carried out in the previous exercise and provide a brief justification for these steps. 

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points:4}

**Your tasks:**
1. Create a topic model on the preprocessed data using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html). Pick a reasonable number for `n_components`, i.e., number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_3
    
</div>

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring word topic association
rubric={points:5}

**Your tasks:**
1. Show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_4
    
</div>

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.5 Exploring document topic association
rubric={points:5}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`. 
2. Comment on the document topic assignment of the model. 

<div class="alert alert-warning">

Solution_2_5
    
</div>

<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points:6}

1. In lecture 19, we talked about multi-class classification. Comment on how each model in the list below might be handling multiclass classification. Check `scikit-learn` documentation for each of these models when you answer this question.  
    - Decision Tree
    - KNN
    - Random Forest    
    - Logistic Regression
    - SVM RBF
2. What is transfer learning in natural language processing or computer vision? Briefly explain.     
3. In Lecture 19 we briefly discussed how neural networks are sort of like `Pipeline`s, in the sense that they involve multiple sequential transformations of the data, finally resulting in the prediction. Why was this property useful when it came to transfer learning?

<div class="alert alert-warning">

Solution_3
    
</div>

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br><br><br>

**PLEASE READ BEFORE YOU SUBMIT:** 

When you are ready to submit your assignment do the following:

1. Run all cells in your notebook to make sure there are no errors by doing `Kernel -> Restart Kernel and Clear All Outputs` and then `Run -> Run All Cells`. 
2. Notebooks with cell execution numbers out of order or not starting from "1" will have marks deducted. Notebooks without the output displayed may not be graded at all (because we need to see the output in order to grade your work).
3. Upload the assignment using Gradescope's drag and drop tool. Check out this [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/) if you need help with Gradescope submission. 
4. Make sure that the plots and output are rendered properly in your submitted file. If the .ipynb file is too big and doesn't render on Gradescope, also upload a pdf or html in addition to the .ipynb so that the TAs can view your submission on Gradescope. 

![](img/eva-well-done.png)